# 태그 기반 클러스터링

## 배경 및 문제 정의

`대표장르추출.ipynb`에서 Steam `genres` 필드 기반으로 게임당 대표 장르를 배정했으나, 다음과 같은 한계가 있었다.

- Steam `genres` 필드는 알파벳 순 멀티레이블 리스트로, 게임의 실제 플레이 스타일을 반영하지 않음

이를 보완하기 위해 사용자가 직접 붙인 **Steam 태그**를 활용한 비지도 클러스터링으로 게임 유형 그룹(`game_type_group`)을 별도 추출한다. 태그는 실제 플레이 경험에 기반한 크라우드소싱 레이블이므로 `genres` 필드보다 게임 성격을 더 잘 반영한다.

## 목적

태그 벡터 유사도 기반 비지도 학습으로 비슷한 게임끼리 묶어, `genres` 필드에 의존하지 않는 대표 장르 분류 기준을 마련한다.

## 분석 흐름

1. 데이터 로드 및 태그 탐색
2. 태그 전처리 (Steam 공식 장르 필터링 → 고빈도 제거 → 저분산 제거)
3. TF-IDF 벡터화 + L2 정규화
4. UMAP 차원 축소 (10차원)
5. HDBSCAN 클러스터링
6. 노이즈 게임 soft clustering으로 재배정
7. 클러스터 해석 및 검증
8. 결과 저장

In [222]:
import json
import warnings
from collections import Counter

import hdbscan
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import umap
from plotly.subplots import make_subplots
from sklearn.preprocessing import normalize
pd.set_option('display.max_columns', None)

warnings.filterwarnings('ignore')

print("라이브러리 로드 완료")

라이브러리 로드 완료


## 데이터 로드

`steam_indie_tags.csv`와 `steam_indie_9692.csv`를 불러와 `appid` 기준으로 inner join합니다. 태그 정보가 있는 게임만 클러스터링 대상으로 사용합니다.

In [223]:
df_tags = pd.read_csv('../../../data/processed/steam_indie_tags.csv')
df_main = pd.read_csv('../../../data/processed/steam_indie_9692.csv')

df = pd.merge(df_main[['appid']], df_tags, on='appid', how='inner')

print(f"steam_indie_tags:  {len(df_tags)}개")
print(f"steam_indie_9692:  {len(df_main)}개")
print(f"inner join 결과:   {len(df)}개 (클러스터링 대상)")

steam_indie_tags:  9706개
steam_indie_9692:  9692개
inner join 결과:   9692개 (클러스터링 대상)


## 1. 태그 탐색

### Steam 공식 장르 태그 정의

Steam 태그는 사용자가 자유롭게 붙이는 크라우드소싱 레이블이므로 노이즈가 많습니다. 클러스터링의 의미 있는 구분을 위해 Steam 공식 분류 체계의 장르 태그(Top-Level Genres / Genres / Sub-Genres)만 사용합니다.

In [224]:
# Steam 공식 Top-Level Genres, Genres, Sub-Genres만 클러스터링 대상으로 사용
STEAM_GENRES = set([
    # Top-Level Genres
    'Action', 'Adventure', 'Casual', 'Experimental', 'Puzzle', 'Racing',
    'RPG', 'Simulation', 'Sports', 'Strategy', 'Tabletop',

    # Genres
    'Action RPG', 'Action-Adventure', 'Arcade', 'Auto Battler', 'Automobile Sim',
    'Base-Building', 'Baseball', 'Basketball', 'Battle Royale', 'BMX', 'Board Game',
    'Bowling', 'Building', 'Card Game', 'Character Action Game', 'Chess', 'Clicker',
    'Cycling', 'Diplomacy', 'e-sports', 'Exploration', 'Farming Sim', 'Fighting',
    'God Game', 'Golf', 'Hacking', 'Hidden Object', 'Hockey', 'Idler',
    'Interactive Fiction', 'Management', 'Match 3', 'Medical Sim', 'Mini Golf',
    'Mining', 'MMORPG', 'MOBA', 'Motocross', 'Open World', 'Outbreak Sim',
    'Party-Based RPG', 'Pinball', 'Platformer', 'Point & Click', 'Rhythm', 'Rogue-like',
    'RTS', 'Sandbox', 'Shooter', 'Skateboarding', 'Skating', 'Skiing', 'Snowboarding',
    'Space Sim', 'Stealth', 'Strategy RPG', 'Survival', 'Tennis', 'Tower Defense',
    'Trivia', 'Turn-Based Strategy', 'Visual Novel', 'Walking Simulator', 'Word Game',
    'Wrestling',

    # Sub-Genres
    '2D Fighter', '2D Platformer', '3D Fighter', '3D Platformer', '4X',
    'Action Roguelike', 'Arena Shooter', "Beat 'em up", 'Bullet Hell', 'Card Battler',
    'Choose Your Own Adventure', 'City Builder', 'Collectathon', 'Colony Sim',
    'Combat Racing', 'CRPG', 'Dating Sim', 'Dungeon Crawler', 'Education', 'Flight',
    'FPS', 'Grand Strategy', 'Hack and Slash', 'Heist', 'Hero Shooter', 'Horror',
    'Immersive Sim', 'Investigation', 'JRPG', 'Life Sim', 'Looter Shooter',
    'Metroidvania', 'Mystery Dungeon', 'On-Rails Shooter', 'Open World Survival Craft',
    'Political Sim', 'Precision Platformer', 'Programming', 'Real Time Tactics',
    'Roguelike Deckbuilder', 'Roguelite', 'Rogue-lite', 'Roguevania', 'Runner',
    "Shoot 'Em Up", 'Side Scroller', 'Sokoban', 'Solitaire', 'Souls-like',
    'Spectacle fighter', 'Spelling', 'Survival Horror', 'Tactical RPG',
    'Third-Person Shooter', 'Time Management', 'Top-Down Shooter', 'Trading',
    'Trading Card Game', 'Traditional Roguelike', 'Turn-Based Tactics',
    'Twin Stick Shooter', 'Typing', 'Wargame', 'Deckbuilding',
])

print(f"Steam 공식 장르 태그 수: {len(STEAM_GENRES)}")

Steam 공식 장르 태그 수: 140


### Steam 공식 장르 태그 출현 빈도 확인

전체 게임 중 각 태그가 몇 %의 게임에 등장하는지 확인합니다. 이후 고빈도 태그 제거 기준을 결정하는 데 사용합니다.

In [225]:
# [태그 탐색] Steam 공식 장르 태그 출현 빈도 확인
tag_game_count = Counter()
for tags_str in df['tags'].dropna():
    tags_dict = json.loads(tags_str)
    for tag, votes in tags_dict.items():
        if tag in STEAM_GENRES:
            tag_game_count[tag] += 1

total_games = len(df)
tag_freq = pd.DataFrame({
    'tag': list(tag_game_count.keys()),
    'game_count': list(tag_game_count.values())
})
tag_freq['ratio'] = tag_freq['game_count'] / total_games
tag_freq = tag_freq.sort_values('ratio', ascending=False).reset_index(drop=True)

fig = go.Figure()
fig.add_trace(go.Bar(x=tag_freq['tag'], y=tag_freq['ratio'], name='출현 비율'))
fig.update_layout(
    title='Steam 공식 장르 태그 출현 빈도',
    xaxis_title='장르 태그',
    yaxis_title='출현 비율',
    height=500,
    xaxis_tickangle=-45
)
fig.show()

print(f"사용할 장르 태그 수: {len(tag_freq)}")
print(tag_freq[['tag', 'ratio', 'game_count']].to_string(index=False))

사용할 장르 태그 수: 139
                      tag    ratio  game_count
                Adventure 0.478745        4640
                   Casual 0.430458        4172
                   Action 0.414466        4017
              Exploration 0.284049        2753
               Simulation 0.255881        2480
                   Puzzle 0.237103        2298
                      RPG 0.224412        2175
                   Horror 0.206149        1998
                 Strategy 0.202848        1966
         Action-Adventure 0.182522        1769
                   Arcade 0.139600        1353
               Rogue-lite 0.121544        1178
               Platformer 0.119790        1161
               Rogue-like 0.119686        1160
            2D Platformer 0.106686        1034
                  Shooter 0.102352         992
             Visual Novel 0.096059         931
                 Survival 0.094305         914
        Walking Simulator 0.093995         911
            Immersive Sim 0.087082         

### [탐색] Genres 카테고리만 사용 시 태그 분포 확인

Top-Level Genres를 제외한 Genres 카테고리 태그만 사용했을 때 게임당 평균 태그 수와 분포를 확인합니다. 태그가 너무 적으면 클러스터링 정보량이 부족하므로 사전 검토합니다.

In [226]:
# [탐색] Genres만 사용했을 때 게임당 평균 태그 수 확인
STEAM_GENRES_ONLY = set([
    'Action RPG', 'Action-Adventure', 'Arcade', 'Auto Battler', 'Automobile Sim',
    'Base-Building', 'Baseball', 'Basketball', 'Battle Royale', 'BMX', 'Board Game',
    'Bowling', 'Building', 'Card Game', 'Character Action Game', 'Chess', 'Clicker',
    'Cycling', 'Diplomacy', 'e-sports', 'Exploration', 'Farming Sim', 'Fighting',
    'God Game', 'Golf', 'Hacking', 'Hidden Object', 'Hockey', 'Idler',
    'Interactive Fiction', 'Management', 'Match 3', 'Medical Sim', 'Mini Golf',
    'Mining', 'MMORPG', 'MOBA', 'Motocross', 'Open World', 'Outbreak Sim',
    'Party-Based RPG', 'Pinball', 'Platformer', 'Point & Click', 'Rhythm', 'Rogue-like',
    'RTS', 'Sandbox', 'Shooter', 'Skateboarding', 'Skating', 'Skiing', 'Snowboarding',
    'Space Sim', 'Stealth', 'Strategy RPG', 'Survival', 'Tennis', 'Tower Defense',
    'Trivia', 'Turn-Based Strategy', 'Visual Novel', 'Walking Simulator', 'Word Game',
    'Wrestling',
])

# 빈도 계산
tag_count_genres_only = Counter()
for tags_str in df['tags'].dropna():
    for tag in json.loads(tags_str).keys():
        if tag in STEAM_GENRES_ONLY:
            tag_count_genres_only[tag] += 1

total_games = len(df)

# 빈도 20% 이상 제거
freq_blacklist_genres = {tag for tag, cnt in tag_count_genres_only.items() if cnt / total_games >= 0.20}
print(f"Genres 카테고리 태그 수: {len(STEAM_GENRES_ONLY)}")
print(f"\n빈도 20% 이상 제거 태그 ({len(freq_blacklist_genres)}개):")
for tag in sorted(freq_blacklist_genres, key=lambda t: -tag_count_genres_only[t]):
    print(f"  {tag}: {tag_count_genres_only[tag]/total_games*100:.1f}%")

# 게임당 태그 수 분포
rows_genres = []
for tags_str in df['tags'].dropna():
    tags_dict = json.loads(tags_str)
    filtered = {tag: votes for tag, votes in tags_dict.items()
                if tag in STEAM_GENRES_ONLY and tag not in freq_blacklist_genres}
    rows_genres.append(filtered)

tag_matrix_genres = pd.DataFrame(rows_genres).fillna(0)
tags_per_game = (tag_matrix_genres > 0).sum(axis=1)

print(f"\n태그 행렬 shape: {tag_matrix_genres.shape}")
print(f"게임당 평균 장르 태그 수: {tags_per_game.mean():.2f}개")
print(f"게임당 중앙값: {tags_per_game.median():.1f}개")
print(f"태그 0개 게임 수: {(tags_per_game == 0).sum()}개 ({(tags_per_game == 0).mean()*100:.1f}%)")
print(f"태그 1개 게임 수: {(tags_per_game == 1).sum()}개")
print(f"태그 2개 이하 게임 수: {(tags_per_game <= 2).sum()}개 ({(tags_per_game <= 2).mean()*100:.1f}%)")
print(f"태그 3개 이상 게임 수: {(tags_per_game >= 3).sum()}개 ({(tags_per_game >= 3).mean()*100:.1f}%)")

fig = go.Figure()
fig.add_trace(go.Histogram(x=tags_per_game, nbinsx=20, name='게임 수'))
fig.update_layout(
    title='Genres만 사용 시 게임당 태그 수 분포 (20% 빈도 제거 후)',
    xaxis_title='태그 수',
    yaxis_title='게임 수',
    height=400
)
fig.show()

Genres 카테고리 태그 수: 65

빈도 20% 이상 제거 태그 (1개):
  Exploration: 28.4%

태그 행렬 shape: (9692, 64)
게임당 평균 장르 태그 수: 2.13개
게임당 중앙값: 2.0개
태그 0개 게임 수: 786개 (8.1%)
태그 1개 게임 수: 2639개
태그 2개 이하 게임 수: 6484개 (66.9%)
태그 3개 이상 게임 수: 3208개 (33.1%)


## 2. 태그 전처리

### 전처리 1 — 고빈도 태그 제거

출현 비율 **20% 이상** 태그는 너무 범용적이어서 클러스터 구분력이 없으므로 제거합니다.

> Adventure(47.9%), Casual(43%), Action(41.4%), Exploration(28.4%) 등

In [227]:
# [전처리 1] Steam 공식 장르 태그만 남기되 빈도 20% 이상 태그 추가 제거
# 20% 이상: Adventure(47.9%), Casual(43%), Action(41.4%), Exploration(28.4%) 등
# → 너무 흔해서 클러스터 구분력 없는 태그 제거

FREQ_THRESHOLD = 0.20

tag_game_count_genre = Counter()
for tags_str in df['tags'].dropna():
    for tag in json.loads(tags_str).keys():
        if tag in STEAM_GENRES:
            tag_game_count_genre[tag] += 1

total_games = len(df)
freq_blacklist = {tag for tag, cnt in tag_game_count_genre.items() if cnt / total_games >= FREQ_THRESHOLD}

print(f"빈도 {FREQ_THRESHOLD*100:.0f}% 이상 제거 태그 ({len(freq_blacklist)}개):")
for tag in sorted(freq_blacklist, key=lambda t: -tag_game_count_genre[t]):
    print(f"  {tag}: {tag_game_count_genre[tag]/total_games*100:.1f}%")

rows = []
for tags_str in df['tags'].dropna():
    tags_dict = json.loads(tags_str)
    rows.append({tag: votes for tag, votes in tags_dict.items()
                 if tag in STEAM_GENRES and tag not in freq_blacklist})

tag_matrix = pd.DataFrame(rows).fillna(0)
print(f"\n태그 행렬 shape: {tag_matrix.shape}")
print(f"게임당 평균 장르 태그 수: {(tag_matrix > 0).sum(axis=1).mean():.1f}개")

빈도 20% 이상 제거 태그 (9개):
  Adventure: 47.9%
  Casual: 43.0%
  Action: 41.4%
  Exploration: 28.4%
  Simulation: 25.6%
  Puzzle: 23.7%
  RPG: 22.4%
  Horror: 20.6%
  Strategy: 20.3%

태그 행렬 shape: (9692, 130)
게임당 평균 장르 태그 수: 4.1개


### 전처리 2 — 태그별 분산 계산

태그 행렬의 컬럼별 분산을 계산합니다. 분산이 낮은 태그는 게임 간 차이를 설명하지 못하므로 후속 단계에서 제거 대상이 됩니다.

In [228]:
# [전처리 2] 태그별 분산 계산 및 시각화
tag_variance = tag_matrix.var(axis=0).sort_values(ascending=False).reset_index()
tag_variance.columns = ['tag', 'variance']
top50 = tag_variance.head(50)

fig = make_subplots(rows=1, cols=2, subplot_titles=('전체 장르 태그 분산 분포', '분산 상위 50개 태그'))
fig.add_trace(go.Scatter(y=tag_variance['variance'].values, mode='lines', name='분산'), row=1, col=1)
fig.add_trace(go.Bar(x=top50['variance'][::-1], y=top50['tag'][::-1], orientation='h', name='분산'), row=1, col=2)
fig.update_xaxes(title_text='태그 순위 (분산 높은 순)', row=1, col=1)
fig.update_yaxes(title_text='분산', row=1, col=1)
fig.update_xaxes(title_text='분산', row=1, col=2)
fig.update_layout(title='장르 태그별 분산 분포 (클러스터링 기여도)', height=600, showlegend=False)
fig.show()

print(f"전체 장르 태그 수: {len(tag_variance)}")
print("\n분산 상위 30개 태그:")
print(tag_variance.head(30).to_string(index=False))

전체 장르 태그 수: 130

분산 상위 30개 태그:
                      tag     variance
                  Sandbox 92085.885637
               Open World 83181.667852
                 Building 63795.855450
              Farming Sim 63408.909627
                  Shooter 63167.698994
            Base-Building 60185.966867
      Turn-Based Strategy 60164.925640
                 Life Sim 59247.906876
                 Survival 46487.039160
       Turn-Based Tactics 45790.820834
             Metroidvania 44744.631625
                     CRPG 43308.654264
Open World Survival Craft 38409.345694
         Top-Down Shooter 37225.999606
       Twin Stick Shooter 36017.015640
               Rogue-lite 33440.604014
            2D Platformer 30894.366696
             Collectathon 22859.408945
            Arena Shooter 20878.165169
                     JRPG 20759.411413
                  Stealth 20440.883658
         Action-Adventure 20089.700169
                Card Game 15699.861617
           Looter Shooter 14579.4

### 전처리 3 — 분산 하위 25% 태그 제거 후보 확인

분산 하위 25% 분위수를 기준으로 제거할 태그 목록을 사전 확인합니다.

In [229]:
# [전처리 3] 분산 하위 25% 태그 제거 후보 확인
variance_threshold = tag_variance['variance'].quantile(0.25)
low_variance_tags = tag_variance[tag_variance['variance'] <= variance_threshold]

print(f"분산 하위 25% threshold: {variance_threshold:.2f}")
print(f"제거 후보 태그 수: {len(low_variance_tags)}")
print(low_variance_tags[['tag', 'variance']].to_string(index=False))

분산 하위 25% threshold: 260.95
제거 후보 태그 수: 33
                  tag   variance
               MMORPG 258.967863
           Roguevania 251.877457
                Heist 236.446593
    Trading Card Game 231.016106
Character Action Game 221.722168
Traditional Roguelike 221.176758
            Mini Golf 219.245656
                 Golf 210.492932
              Match 3 195.516622
               Typing 195.236316
              Hacking 172.895062
         Outbreak Sim 161.888462
          Programming 159.587593
            Solitaire 148.344630
               Trivia 143.720344
                Chess 141.308948
           Basketball 100.250358
              Skating  90.633720
             Spelling  88.804885
               Skiing  78.686350
        Skateboarding  75.880250
            Wrestling  74.445870
                 MOBA  55.526503
          Medical Sim  39.981116
         Snowboarding  39.049483
              Cycling  35.636211
              Pinball  30.333287
               Tennis  29.306301


### 전처리 4 — 최종 태그 행렬 확정

분산 하위 25% 태그를 제거하여 클러스터링에 사용할 최종 태그 행렬을 확정합니다.

In [230]:
# [전처리 4] 분산 하위 25% 제거 → 최종 태그 행렬 확정
high_variance_tags = tag_variance[tag_variance['variance'] > variance_threshold]['tag'].tolist()
tag_matrix_filtered = tag_matrix[high_variance_tags]

print(f"최종 태그 수: {tag_matrix_filtered.shape[1]}")
print(f"게임 수: {tag_matrix_filtered.shape[0]}")

최종 태그 수: 97
게임 수: 9692


## 3. 벡터화

태그 투표수 행렬에 **TF-IDF 가중치**를 적용한 뒤 **L2 정규화**합니다.

- TF: 게임 내 태그 투표수 비율
- IDF: 희귀 태그에 가중치 부여
- L2 정규화: 게임 간 벡터 크기 차이 제거

In [231]:
# [벡터화] TF-IDF + L2 정규화
X = tag_matrix_filtered.values.astype(float)

row_sums = X.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
tf = X / row_sums

n_games = X.shape[0]
df_count = (X > 0).sum(axis=0)
idf = np.log((n_games + 1) / (df_count + 1)) + 1

tfidf_normalized = normalize(tf * idf, norm='l2')
print(f"TF-IDF 행렬 shape: {tfidf_normalized.shape}")

TF-IDF 행렬 shape: (9692, 97)


## 4. 차원 축소 — UMAP

고차원 TF-IDF 벡터를 **UMAP으로 10차원**으로 축소합니다. 클러스터링용이므로 `min_dist=0.0`으로 설정하여 군집 구조를 최대한 보존합니다.

| 파라미터 | 값 | 설명 |
|---|---|---|
| `n_components` | 10 | 클러스터링용 차원 수 |
| `n_neighbors` | 15 | 로컬 구조 보존 범위 |
| `min_dist` | 0.0 | 군집 응집도 최대화 |
| `metric` | cosine | 태그 벡터 유사도 측정 |

In [232]:
# [차원 축소] UMAP 10차원 (클러스터링용)
reducer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
embedding = reducer.fit_transform(tfidf_normalized)
print(f"UMAP 임베딩 shape: {embedding.shape}")

UMAP 임베딩 shape: (9692, 10)


## 5. 클러스터링 — HDBSCAN

### 파라미터 탐색

`min_cluster_size`를 변화시키며 클러스터 수와 노이즈 비율의 trade-off를 확인합니다.

In [233]:
# [파라미터 탐색] min_cluster_size별 클러스터 수 vs 노이즈 비율
results = []
for mcs in [30, 50, 70, 100, 130, 150]:
    c = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=10,
        metric='euclidean',
        cluster_selection_method='eom'
    ).fit_predict(embedding)
    n_clusters = len(set(c)) - (1 if -1 in c else 0)
    n_noise = (c == -1).sum()
    results.append({'min_cluster_size': mcs, '클러스터 수': n_clusters, '노이즈 수': n_noise, '노이즈 비율(%)': round(n_noise / len(c) * 100, 1)})

result_df = pd.DataFrame(results)
print(result_df.to_string(index=False))

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Scatter(x=result_df['min_cluster_size'], y=result_df['클러스터 수'], mode='lines+markers', name='클러스터 수', line=dict(color='blue')),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=result_df['min_cluster_size'], y=result_df['노이즈 비율(%)'], mode='lines+markers', name='노이즈 비율(%)', line=dict(color='red', dash='dash')),
    secondary_y=True
)
fig.update_layout(title='min_cluster_size에 따른 클러스터 수 vs 노이즈 비율', xaxis_title='min_cluster_size', height=450)
fig.update_yaxes(title_text='클러스터 수', secondary_y=False)
fig.update_yaxes(title_text='노이즈 비율(%)', secondary_y=True)
fig.show()

 min_cluster_size  클러스터 수  노이즈 수  노이즈 비율(%)
               30     123   1309       13.5
               50      73   1486       15.3
               70      42   1142       11.8
              100      33   1173       12.1
              130      25   1611       16.6
              150      19   1689       17.4


### HDBSCAN 클러스터링 실행

탐색 결과를 바탕으로 `min_cluster_size=130`으로 최종 클러스터링을 수행합니다.

In [234]:
# [클러스터링] HDBSCAN — min_cluster_size는 위 탐색 결과 보고 설정
MIN_CLUSTER_SIZE = 130

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
labels = clusterer.fit_predict(embedding)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()
print(f"클러스터 수: {n_clusters}")
print(f"노이즈(미분류) 게임 수: {n_noise} ({n_noise/len(labels)*100:.1f}%)")
for c in sorted(set(labels)):
    label = '노이즈' if c == -1 else f'클러스터 {c}'
    print(f"  {label}: {(labels == c).sum()}개")

클러스터 수: 25
노이즈(미분류) 게임 수: 1611 (16.6%)
  노이즈: 1611개
  클러스터 0: 274개
  클러스터 1: 148개
  클러스터 2: 215개
  클러스터 3: 282개
  클러스터 4: 210개
  클러스터 5: 359개
  클러스터 6: 330개
  클러스터 7: 260개
  클러스터 8: 294개
  클러스터 9: 1149개
  클러스터 10: 844개
  클러스터 11: 419개
  클러스터 12: 182개
  클러스터 13: 292개
  클러스터 14: 221개
  클러스터 15: 220개
  클러스터 16: 147개
  클러스터 17: 140개
  클러스터 18: 326개
  클러스터 19: 418개
  클러스터 20: 380개
  클러스터 21: 260개
  클러스터 22: 292개
  클러스터 23: 150개
  클러스터 24: 269개


### 노이즈 게임 재배정 — Soft Clustering

HDBSCAN에서 노이즈(-1)로 분류된 게임을 **soft clustering**으로 가장 소속 확률이 높은 클러스터에 배정합니다. 노이즈를 분석에서 제외하지 않고 최대한 활용하기 위한 처리입니다.

In [235]:
# [클러스터링] 노이즈 게임 soft clustering으로 가장 가까운 클러스터에 배정
soft_clusters = hdbscan.all_points_membership_vectors(clusterer)

# 노이즈(-1) 게임만 가장 높은 확률의 클러스터로 배정
labels_filled = labels.copy()
noise_mask = labels == -1
labels_filled[noise_mask] = soft_clusters[noise_mask].argmax(axis=1)

n_noise_remaining = (labels_filled == -1).sum()
print(f"배정 전 노이즈: {(labels == -1).sum()}개")
print(f"배정 후 노이즈: {n_noise_remaining}개")
print(f"\n클러스터별 게임 수 (노이즈 배정 후):")
for c in sorted(set(labels_filled)):
    print(f"  클러스터 {c}: {(labels_filled == c).sum()}개")

배정 전 노이즈: 1611개
배정 후 노이즈: 0개

클러스터별 게임 수 (노이즈 배정 후):
  클러스터 0: 274개
  클러스터 1: 148개
  클러스터 2: 215개
  클러스터 3: 440개
  클러스터 4: 210개
  클러스터 5: 398개
  클러스터 6: 423개
  클러스터 7: 260개
  클러스터 8: 294개
  클러스터 9: 1345개
  클러스터 10: 844개
  클러스터 11: 586개
  클러스터 12: 357개
  클러스터 13: 303개
  클러스터 14: 293개
  클러스터 15: 386개
  클러스터 16: 167개
  클러스터 17: 179개
  클러스터 18: 380개
  클러스터 19: 576개
  클러스터 20: 383개
  클러스터 21: 304개
  클러스터 22: 330개
  클러스터 23: 194개
  클러스터 24: 403개


## 6. 시각화

UMAP을 **2차원**으로 재축소하여 클러스터 분포를 시각화합니다. `×` 마커는 원래 노이즈였다가 soft clustering으로 재배정된 게임입니다.

In [236]:
# [시각화] UMAP 2D로 클러스터 분포 확인 (노이즈 배정 후)
import plotly.express as px

reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
embedding_2d = reducer_2d.fit_transform(tfidf_normalized)

df_plot = pd.DataFrame({
    'x': embedding_2d[:, 0],
    'y': embedding_2d[:, 1],
    'cluster': labels_filled.astype(str),
    'original': labels.astype(str),
    'name': df[df['tags'].notna()].reset_index(drop=True)['name']
})
df_plot['noise_assigned'] = labels == -1

fig = px.scatter(
    df_plot, x='x', y='y', color='cluster',
    symbol=df_plot['noise_assigned'].map({True: 'x', False: 'circle'}),
    hover_data=['name', 'cluster', 'noise_assigned'],
    title=f'UMAP + HDBSCAN 클러스터링 결과 ({n_clusters}개 클러스터, 노이즈 배정 포함)',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    height=700
)
fig.update_traces(marker=dict(size=3, opacity=0.6))
fig.show()

## 7. 클러스터 해석

클러스터별 상위 10개 태그를 집계하여 각 클러스터가 어떤 유형의 게임 그룹인지 해석합니다.

In [237]:
# [결과 해석] 클러스터별 대표 태그 출력 (노이즈 배정 후)
df_result = df[df['tags'].notna()].reset_index(drop=True).copy()
df_result['cluster'] = labels_filled

print("=== 클러스터별 상위 태그 (장르 그룹 해석) ===\n")
for c in sorted(set(labels_filled)):
    cluster_games = df_result[df_result['cluster'] == c]
    tag_sum = Counter()
    for tags_str in cluster_games['tags']:
        for tag, votes in json.loads(tags_str).items():
            if tag in high_variance_tags:
                tag_sum[tag] += votes
    top_tags = [t for t, _ in tag_sum.most_common(10)]
    print(f"[클러스터 {c}] ({len(cluster_games)}개 게임)")
    print(f"  대표 태그: {', '.join(top_tags)}\n")

=== 클러스터별 상위 태그 (장르 그룹 해석) ===

[클러스터 0] (274개 게임)
  대표 태그: 

[클러스터 1] (148개 게임)
  대표 태그: Walking Simulator, Point & Click, Management, Action-Adventure, Visual Novel, Open World, 2D Platformer, Survival Horror, Flight, Life Sim

[클러스터 2] (215개 게임)
  대표 태그: Dating Sim, Visual Novel, Interactive Fiction, Choose Your Own Adventure, Immersive Sim, Life Sim, Point & Click, 2D Platformer, CRPG, Clicker

[클러스터 3] (440개 게임)
  대표 태그: Sports, Racing, Arcade, Automobile Sim, Open World, Sandbox, Action-Adventure, Combat Racing, Immersive Sim, Flight

[클러스터 4] (210개 게임)
  대표 태그: Metroidvania, 2D Platformer, Platformer, Action-Adventure, Shooter, Side Scroller, Rogue-lite, Souls-like, Stealth, Base-Building

[클러스터 5] (398개 게임)
  대표 태그: 3D Platformer, Platformer, Action-Adventure, Arcade, Precision Platformer, Runner, Collectathon, Racing, Walking Simulator, Open World

[클러스터 6] (423개 게임)
  대표 태그: 2D Platformer, Platformer, Side Scroller, Action-Adventure, Rhythm, Arcade, Shooter, Walking Simulator

## 8. 결과 저장

클러스터 배정 결과를 `game_type_group` 컬럼으로 `steam_indie_9692.csv`에 추가하여 `steam_indie_9692_with_group.csv`로 저장합니다.

In [238]:
# [최종 출력] 클러스터 결과를 steam_indie_9692.csv에 game_type_group 컬럼으로 추가
df_result_indexed = df[['appid']].reset_index(drop=True).copy()
df_result_indexed['game_type_group'] = labels_filled

# steam_indie_9692와 join
df_output = pd.merge(df_main, df_result_indexed, on='appid', how='left')

print(f"전체 게임 수: {len(df_output)}")
print(f"game_type_group 배정 완료: {df_output['game_type_group'].notna().sum()}개")
print(f"game_type_group 미배정: {df_output['game_type_group'].isna().sum()}개")
print()
print("게임 유형 그룹별 게임 수:")
print(df_output['game_type_group'].value_counts().sort_index().to_string())

df_output.to_csv('../../../data/processed/steam_indie_9692_with_group.csv', index=False)
print("\nsteam_indie_9692_with_group.csv 저장 완료")

전체 게임 수: 9692
game_type_group 배정 완료: 9692개
game_type_group 미배정: 0개

게임 유형 그룹별 게임 수:
game_type_group
0      274
1      148
2      215
3      440
4      210
5      398
6      423
7      260
8      294
9     1345
10     844
11     586
12     357
13     303
14     293
15     386
16     167
17     179
18     380
19     576
20     383
21     304
22     330
23     194
24     403

steam_indie_9692_with_group.csv 저장 완료


## 9. 검증

### 검증 1 — 실루엣 스코어 (정량적 검증)

클러스터링 품질을 정량적으로 측정합니다.

| 지표 | 기준 |
|---|---|
| 실루엣 스코어 | 1에 가까울수록 좋음, **0.5 이상이면 양호** |
| Davies-Bouldin 스코어 | 0에 가까울수록 좋음 |

In [239]:
# [검증 1] 실루엣 스코어 (정량적 검증)
from sklearn.metrics import silhouette_score, davies_bouldin_score

# 계산 비용이 높아 샘플링 후 측정
sample_size = 2000
score_silhouette = silhouette_score(embedding, labels_filled, sample_size=sample_size, random_state=42)
score_db = davies_bouldin_score(embedding, labels_filled)

print(f"실루엣 스코어: {score_silhouette:.4f}  (1에 가까울수록 좋음, 0.5 이상이면 양호)")
print(f"Davies-Bouldin 스코어: {score_db:.4f}  (0에 가까울수록 좋음)")
print()

# 클러스터별 실루엣 스코어
from sklearn.metrics import silhouette_samples
sample_idx = np.random.RandomState(42).choice(len(embedding), size=sample_size, replace=False)
silhouette_vals = silhouette_samples(embedding[sample_idx], labels_filled[sample_idx])

cluster_scores = pd.DataFrame({
    'cluster': labels_filled[sample_idx],
    'silhouette': silhouette_vals
}).groupby('cluster')['silhouette'].mean().sort_values(ascending=False)

fig = go.Figure(go.Bar(
    x=cluster_scores.index.astype(str),
    y=cluster_scores.values,
    marker_color=['green' if v >= 0.5 else 'orange' if v >= 0.3 else 'red' for v in cluster_scores.values]
))
fig.add_hline(y=0.5, line_dash='dash', line_color='green', annotation_text='양호(0.5)')
fig.add_hline(y=0.3, line_dash='dash', line_color='orange', annotation_text='보통(0.3)')
fig.update_layout(title='클러스터별 실루엣 스코어', xaxis_title='클러스터', yaxis_title='실루엣 스코어', height=400)
fig.show()

print("클러스터별 실루엣 스코어:")
print(cluster_scores.to_string())

실루엣 스코어: 0.3797  (1에 가까울수록 좋음, 0.5 이상이면 양호)
Davies-Bouldin 스코어: 1.1272  (0에 가까울수록 좋음)



클러스터별 실루엣 스코어:
cluster
0     0.993124
1     0.990913
2     0.918362
4     0.838852
23    0.701902
16    0.688353
18    0.654618
20    0.630241
7     0.590154
8     0.573177
13    0.561276
17    0.557660
22    0.527869
21    0.518165
5     0.504770
11    0.438748
10    0.401758
9     0.353164
6     0.321862
19    0.301680
3     0.232008
15    0.209965
14   -0.200453
12   -0.282337
24   -0.559905


### 검증 2 — 알려진 게임으로 직관적 검증

장르가 명확한 유명 게임이 예상과 일치하는 클러스터에 배정되었는지 확인합니다.

In [240]:
# [검증 2] 알려진 게임으로 클러스터 배정 확인 (직관적 검증)
# 2023~2025년 출시 게임 중 장르가 명확한 유명 게임
known_games = {
    'Balatro':              '카드/덱빌딩',
    'Brotato':              '액션 로그라이크',
    'Hades II':             '액션 로그라이크',
    'MiSide':               '비주얼 노벨/공포',
    'Satisfactory':         '오픈월드 생존/경영',
    'Sons Of The Forest':   '오픈월드 생존',
    'Ready or Not':         'FPS/슈터',
    'The Outlast Trials':   '협동 공포',
    'Buckshot Roulette':    '공포',
    'Party Animals':        '멀티플레이 파티',
    'Barotrauma':           '오픈월드 생존',
    'Last Epoch':           '액션 RPG',
    'WEBFISHING':           '캐주얼',
}

df_known = pd.merge(
    df_result[['appid', 'name', 'cluster']],
    df_main[['appid', 'genres']],
    on='appid', how='left'
)

# 클러스터별 대표 태그 매핑 (결과 해석 셀 기준)
cluster_top_tag = {}
for c in sorted(set(labels_filled)):
    cluster_games = df_result[df_result['cluster'] == c]
    tag_sum = Counter()
    for tags_str in cluster_games['tags']:
        for tag, votes in json.loads(tags_str).items():
            if tag in STEAM_GENRES:
                tag_sum[tag] += votes
    if tag_sum:
        cluster_top_tag[c] = tag_sum.most_common(3)

print(f"{'게임명':<25} {'예상 유형':<22} {'클러스터':<10} {'클러스터 대표 태그'}")
print('-' * 95)
for game, expected in known_games.items():
    match = df_known[df_known['name'].str.contains(game, case=False, na=False)]
    if len(match) > 0:
        row = match.iloc[0]
        c = int(row['cluster'])
        top3 = ', '.join([t for t, _ in cluster_top_tag.get(c, [])])
        print(f"{game:<25} {expected:<22} 클러스터 {c:<5} {top3}")
    else:
        print(f"{game:<25} {expected:<22} {'데이터 없음'}")

게임명                       예상 유형                  클러스터       클러스터 대표 태그
-----------------------------------------------------------------------------------------------
Balatro                   카드/덱빌딩                 클러스터 23    Strategy, Rogue-like, Roguelike Deckbuilder
Brotato                   액션 로그라이크               클러스터 18    Action, Top-Down Shooter, Bullet Hell
Hades II                  액션 로그라이크               클러스터 19    Action, Action Roguelike, Rogue-like
MiSide                    비주얼 노벨/공포              클러스터 10    Simulation, Sandbox, Adventure
Satisfactory              오픈월드 생존/경영             클러스터 10    Simulation, Sandbox, Adventure
Sons Of The Forest        오픈월드 생존                클러스터 10    Simulation, Sandbox, Adventure
Ready or Not              FPS/슈터                 클러스터 16    Action, Shooter, FPS
The Outlast Trials        협동 공포                  클러스터 9     Horror, Adventure, Exploration
Buckshot Roulette         공포                     클러스터 12    Visual Novel, Adventure, Choo